In [1]:
from tensorflow.python.distribute.input_lib import tensor_shape
import os
import keras
import numpy as np
import tensorflow as tf

from keras.preprocessing.image import ImageDataGenerator
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.applications import ResNet50, ResNet50V2, InceptionV3, Xception, ResNet152, ResNet152V2
from keras import Sequential
from keras.layers import Dense
from keras.models import load_model
from keras.layers import Dropout
from keras.layers import GlobalAveragePooling2D

from keras.callbacks import EarlyStopping, ModelCheckpoint

In [2]:
root_path = '//kaggle//input//my-new'
class_names = sorted(os.listdir(root_path))
class_names

['Control', 'Loose']

In [3]:
print(f"Total Number of Classes : {len(class_names)}")

Total Number of Classes : 2


In [4]:
class_dis = [len(os.listdir(root_path + f"/{name}")) for name in class_names]
class_dis

[94, 112]

In [5]:
fig = px.pie(names=class_names, values=class_dis, hole=0.1, title="Class Distribution")
fig.update_layout({'title':{"x":0.5}})
fig.show()

In [6]:
fig = px.bar(x=class_names, y=class_dis, title="Class Distribution")
fig.update_layout({'title':{"x":0.5}})
fig.show()

In [7]:
train_val_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip = True,
    validation_split=0.1
)

test_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

# Load Data
train_ds = train_val_gen.flow_from_directory(root_path, target_size=(256,256), class_mode="binary", batch_size=32, shuffle=True, subset='training')
valid_ds = train_val_gen.flow_from_directory(root_path, target_size=(256,256), class_mode="binary", batch_size=32, shuffle=True, subset='validation')
test_ds = test_gen.flow_from_directory(root_path, target_size=(256,256), class_mode="binary", batch_size=32, subset='validation')

Found 186 images belonging to 2 classes.
Found 20 images belonging to 2 classes.
Found 40 images belonging to 2 classes.


In [8]:
name="ResNet50V2"

base_model = ResNet50V2(include_top=False, weights='imagenet', input_shape=(256,256,3))
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256,activation='relu'),
    Dropout(0.2),
    Dense(len(class_names), activation='softmax')
], name=name)

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


callbacks = [EarlyStopping(patience=5, restore_best_weights=True), ModelCheckpoint(name + ".h5", save_best_only=True)]

94668760/94668760 [==============================] - 1s 0us/step


In [9]:
#Train Model
model.fit(train_ds, epochs=4, validation_data=valid_ds, callbacks=callbacks)

Epoch 1/4
6/6 [==============================] - 31s 4s/step - loss: 1.6012 - accuracy: 0.5430 - val_loss: 0.7066 - val_accuracy: 0.5000
Epoch 2/4
6/6 [==============================] - 24s 4s/step - loss: 0.8591 - accuracy: 0.6129 - val_loss: 0.9802 - val_accuracy: 0.6000
Epoch 3/4
6/6 [==============================] - 25s 4s/step - loss: 0.5973 - accuracy: 0.7097 - val_loss: 0.5066 - val_accuracy: 0.7500
Epoch 4/4
6/6 [==============================] - 24s 4s/step - loss: 0.5169 - accuracy: 0.7151 - val_loss: 0.5703 - val_accuracy: 0.6500


In [10]:
model.summary()

Model: "ResNet50V2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50v2 (Functional)     (None, 8, 8, 2048)        23564800  
                                                                 
 global_average_pooling2d (G  (None, 2048)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 256)               524544    
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 2)                 514       
                                                                 
Total params: 24,089,858
Trainable params: 525,058
Non-trainable params: 23,564,800
______________________________________

In [11]:
model.evaluate(valid_ds)

1/1 [==============================] - 3s 3s/step - loss: 0.6927 - accuracy: 0.7000


[0.6926741600036621, 0.699999988079071]

In [12]:
model.evaluate(test_ds)

2/2 [==============================] - 4s 771ms/step - loss: 0.4601 - accuracy: 0.7750


[0.4601190984249115, 0.7749999761581421]

In [13]:
def show_image(img, title=None):
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')